# Meta-Signal Q4 Gauntlet â€” Unsloth Fine-Tune

Fine-tunes **Llama-3.1-8B-Instruct** on expert demonstrations from the Q4 Gauntlet environment using [Unsloth](https://github.com/unslothai/unsloth) for 2Ã— faster training.

**Runtime**: Nvidia A10G Small (24 GB VRAM) â€” estimated **~12 min** for 10k records, 3 epochs.  
**Dataset**: `Anvit25/meta-signal-expert-demos` on HF Hub (10,250 Alpaca-format records).  
**Output**: LoRA adapter pushed to `Anvit25/meta-signal-q4-agent`.

## Steps
1. Install dependencies
2. Login + load dataset from HF Hub
3. Load Llama-3.1-8B with 4-bit QLoRA via Unsloth
4. Fine-tune with SFTTrainer
5. Inference test
6. Save adapter + push to HF Hub

In [1]:
!pip install --quiet unsloth
!pip install --quiet trl datasets transformers accelerate bitsandbytes peft huggingface_hub
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

GPU: NVIDIA A10G  VRAM: 23.9 GB


In [ ]:
from huggingface_hub import login, hf_hub_download
import json, os

# Paste your HF write token (needs read+write scope)
# Get one at: https://huggingface.co/settings/tokens
HF_TOKEN = ""   # <-- paste your token here, or set HF_TOKEN env var
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
login(token=HF_TOKEN)

# Download dataset from HF Hub
DATA_REPO = "Anvit25/meta-signal-expert-demos"
DATA_PATH = hf_hub_download(
    repo_id   = DATA_REPO,
    filename  = "expert_demos.jsonl",
    repo_type = "dataset",
)
print(f"Dataset downloaded to: {DATA_PATH}")

records = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8")]
print(f"Loaded {len(records):,} records")
phases = {}
for r in records:
    p = r['metadata']['phase']
    phases[p] = phases.get(p, 0) + 1
print("Phase distribution:", phases)

expert_demos.jsonl:   0%|          | 0.00/54.4M [00:00<?, ?B/s]

Dataset downloaded to: /home/user/.cache/huggingface/hub/datasets--Anvit25--meta-signal-expert-demos/snapshots/bcd9c97816fd5907c349e29adfb8999c66ca19cb/expert_demos.jsonl
Loaded 41,000 records
Phase distribution: {'Nominal': 12600, 'Signal_Loss': 13800, 'Andromeda_Glitched': 10800, 'Peak_Load': 3800}


In [3]:
MODEL_NAME   = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # pre-quantised
OUTPUT_DIR   = "meta-signal-lora"
HF_REPO      = "Anvit25/meta-signal-q4-agent"
MAX_SEQ_LEN  = 2048
LORA_RANK    = 16
LORA_ALPHA   = 32
BATCH_SIZE   = 8    # A10G can handle 8 vs T4's 4
GRAD_ACCUM   = 2    # effective batch = 16
EPOCHS       = 1
LR           = 2e-4
WARMUP_RATIO = 0.05

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    lora_alpha                 = LORA_ALPHA,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    lora_dropout               = 0,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)
print(model.print_trainable_parameters())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A10G. Num GPUs = 1. Max memory: 22.301 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
None


In [5]:
from datasets import Dataset

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input \
that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token

def format_record(rec):
    text = ALPACA_PROMPT.format(
        instruction = rec["instruction"],
        input       = rec["input"],
        output      = rec["output"],
    ) + EOS_TOKEN
    return {"text": text}

dataset = Dataset.from_list([format_record(r) for r in records])
print(dataset)
print("\nSample (first 500 chars):\n", dataset[0]["text"][:500])

Dataset({
    features: ['text'],
    num_rows: 41000
})

Sample (first 500 chars):
 Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an advertising budget optimisation agent in Phase 1 (days 1–20) of a 100-day campaign. Signal quality is clean. Your goal is to identify which campaign has the best conversion rate (CVR) and progressively concentrate budget there while staying below the 70% concentration limit to avoid the correlation penalty. Outp


In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    dataset_num_proc   = 4,
    packing            = True,   # pack short sequences for efficiency on A10G
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio                = WARMUP_RATIO,
        num_train_epochs            = EPOCHS,
        learning_rate               = LR,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),  # A10G supports bf16
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 42,
        output_dir                  = OUTPUT_DIR,
        report_to                   = "none",
    ),
)

trainer_stats = trainer.train()
print(f"Training complete. Loss: {trainer_stats.training_loss:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/41000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 41,000 | Num Epochs = 1 | Total steps = 2,563
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.803828
10,2.770048
15,2.614342
20,2.302833
25,1.801717
30,1.186507
35,0.655081
40,0.326652
45,0.217188
50,0.184208


Unsloth: Restored added_tokens_decoder metadata in meta-signal-lora/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in meta-signal-lora/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in meta-signal-lora/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in meta-signal-lora/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in meta-signal-lora/checkpoint-2500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in meta-signal-lora/checkpoint-2563/tokenizer_config.json.


Training complete. Loss: 0.1080


In [7]:
FastLanguageModel.for_inference(model)

test_obs = """\
step=25 day=25 phase=Signal_Loss
budget_remaining=$8500.00 epsilon=16.500
privacy_regime=high_noise learning_status=Optimized
market_trend=Rising regulatory_violation=False
campaigns: camp_feed: spend=$0.0, noisy_conv=0.00, est_roas=0.000, ctr=0.0800, ci=[0.00,0.00] | \
camp_reels: spend=$0.0, noisy_conv=0.00, est_roas=0.000, ctr=0.0700, ci=[0.00,0.00] | \
camp_stories: spend=$0.0, noisy_conv=0.00, est_roas=0.000, ctr=0.0600, ci=[0.00,0.00]"""

prompt = ALPACA_PROMPT.format(
    instruction = (
        "You are an advertising budget optimisation agent in Phase 2 (days 21-50). "
        "iOS ATT has caused 3x noise. Use use_capi=true sparingly. "
        "Hold Phase 1 allocation steady. Output a JSON action."
    ),
    input  = test_obs,
    output = "",
)

inputs  = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):]
print("Model output:")
print(response)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/user/miniconda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/user/miniconda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/hom

Model output:
{"allocations": {"camp_feed": 65.0, "camp_reels": 17.5, "camp_stories": 17.5}, "use_capi": true, "pacing_speed": 1.0, "apply_safety_cap": true}


In [8]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to ./{OUTPUT_DIR}")

model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/{HF_REPO}")
print()
print("Training summary:")
print(f"  Loss:    {trainer_stats.training_loss:.4f}")
print(f"  Runtime: {trainer_stats.metrics.get('train_runtime', 0)/60:.1f} min")
print(f"  Steps:   {trainer_stats.global_step}")

Unsloth: Restored added_tokens_decoder metadata in meta-signal-lora/tokenizer_config.json.


Adapter saved to ./meta-signal-lora


README.md:   0%|          | 0.00/4.28k [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp3rg9liag/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to https://huggingface.co/Anvit25/meta-signal-q4-agent

Training summary:
  Loss:    0.1080
  Runtime: 166.3 min
  Steps:   2563


## Next Steps

1. **Evaluate** the fine-tuned model against the live environment:
   ```bash
   python inference.py --task 7 --model Anvit25/meta-signal-q4-agent
   ```
2. **Compare** score against the ExpertBot baseline (~0.66 on Task 7).
3. **Iterate** â€” generate 500 episodes and re-train for higher quality.
4. **RLHF loop** (optional): use the grader score as reward signal for PPO.